<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/ROSG_Test5Cbis_P1_coupling_compliance_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROSG Test5C-bis — P1 coupling-driven compliance audit

Audit indépendant de Test5C : génération de séquences P1 normalisées sans reconstruction cell-4. Le test compare les tendances de rigidité/compliance sous contraintes avancées et inclut des contrôles.

In [1]:
import os, json, math, zipfile, textwrap, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('/mnt/data/ROSG_Test5Cbis_P1_coupling_compliance_audit')
OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / 'figures'
FIG.mkdir(exist_ok=True)

# --- Constants (CODATA 2018 exact/standard values where relevant) ---
# Used only to document the dimensional P1 kernel; raw dimensionful powers are not used as direct observables.
hbar = 1.054_571_817e-34  # J s
G = 6.674_30e-11          # m^3 kg^-1 s^-2
c = 299_792_458.0         # m s^-1
lP = math.sqrt(hbar*G/c**3)
tP = lP/c
alpha_low = 1/137.035999084
alpha_grid = [1/140, 1/137.035999084, 1/128, 1/118, 1/100]

levels = np.arange(1, 7, dtype=int)

# --- Guardrail philosophy ---
# Test5C-bis is not allowed to reconstruct the cell-4 graph and not allowed to use Test5C outputs
# to create the P1 sequences. It can optionally compare the final trend to Test5C-i6 as an external check.

# P1 normalized amplification model.
# Raw alpha_eff^(i) contains powers of lP^2 and is dimensionful for i>1.
# Therefore the audit uses a dimensionless, i=1-normalized amplification family:
# Gamma_P1(i; beta) = exp(beta*(i-1)*ln(4)).
# beta scans departures from exact dyadic-area amplification without changing the i=1 anchoring.
# This is an audit of trend robustness, not a measurement of alpha_eff.
def gamma_p1(levels, beta=1.0):
    return np.exp(beta * (levels - 1) * np.log(4.0))

# Bounded conductivo-spectral response independent of Test5C graph outputs.
# This encodes the advanced constraint that a response cannot grow combinatorially with the load.
def response_bounded(levels, A=0.58, tau=1.8, alphaP=alpha_low, alpha_ref=alpha_low, alpha_exp=0.0):
    # alpha_exp default 0 means the UV matching factor cancels after i=1 normalization.
    return (1.0 + A*(1.0 - np.exp(-(levels-1)/tau))) * (alphaP/alpha_ref)**alpha_exp

# Alternative controls
def gamma_null(levels):
    return np.ones_like(levels, dtype=float)

def gamma_linear(levels):
    return levels.astype(float)

def gamma_saturating(levels, Gmax=32.0, tau=1.5):
    # Normalized to 1 at i=1, then saturates.
    return 1.0 + (Gmax-1.0)*(1.0 - np.exp(-(levels-1)/tau))

def gamma_random_order(levels, beta=1.0, seed=42):
    vals = gamma_p1(levels, beta=beta).copy()
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(vals))
    # keep i=1 anchored by swapping any position containing first baseline back to index 0
    vals_shuf = vals[perm]
    idx_one = int(np.argmin(vals_shuf))
    vals_shuf[0], vals_shuf[idx_one] = vals_shuf[idx_one], vals_shuf[0]
    return vals_shuf

def metrics_for_sequence(levels, Gamma, RK, model_name, params=None):
    Khat = RK/Gamma
    chihat = Gamma/RK
    df = pd.DataFrame({
        'model': model_name,
        'i': levels,
        'Gamma': Gamma,
        'R_K': RK,
        'K_hat': Khat,
        'chi_hat': chihat,
    })
    if params:
        for k,v in params.items():
            df[k]=v
    # stabilization window i>=3
    win = df[df['i']>=3]
    gamma_mon = bool(np.all(np.diff(win['Gamma']) > 0))
    chi_mon = bool(np.all(np.diff(win['chi_hat']) > 0))
    K_mon = bool(np.all(np.diff(win['K_hat']) < 0))
    RK_bounded = bool((win['R_K'].max()/win['R_K'].min()) < 1.25)
    i5 = df[df['i']==5].iloc[0]
    i6 = df[df['i']==6].iloc[0]
    summary = {
        'model': model_name,
        'Gamma_i6_over_i5': float(i6['Gamma']/i5['Gamma']),
        'R_K_i6_over_i5': float(i6['R_K']/i5['R_K']),
        'K_hat_i6_over_i5': float(i6['K_hat']/i5['K_hat']),
        'chi_hat_i6_over_i5': float(i6['chi_hat']/i5['chi_hat']),
        'Gamma_monotone_i3_i6': gamma_mon,
        'chi_monotone_i3_i6': chi_mon,
        'K_hat_decreasing_i3_i6': K_mon,
        'R_K_bounded_i3_i6': RK_bounded,
        'hyper_compliance_flag': bool(gamma_mon and chi_mon and K_mon and RK_bounded and (i6['chi_hat']/i5['chi_hat']>2.0)),
    }
    if params:
        summary.update(params)
    return df, summary

# Baseline and controls
all_profiles = []
all_summaries = []

baseline_params = {'beta':1.0, 'A':0.58, 'tau':1.8, 'alphaP':alpha_low}
Gamma = gamma_p1(levels, beta=baseline_params['beta'])
RK = response_bounded(levels, A=baseline_params['A'], tau=baseline_params['tau'], alphaP=baseline_params['alphaP'])
df, sm = metrics_for_sequence(levels, Gamma, RK, 'P1_bounded_baseline', baseline_params)
all_profiles.append(df); all_summaries.append(sm)

controls = [
    ('CTRL_null_load', gamma_null(levels), response_bounded(levels)),
    ('CTRL_linear_load', gamma_linear(levels), response_bounded(levels)),
    ('CTRL_saturating_load', gamma_saturating(levels), response_bounded(levels)),
    ('CTRL_random_order_load', gamma_random_order(levels), response_bounded(levels)),
]
for name, Gm, RKm in controls:
    df, sm = metrics_for_sequence(levels, Gm, RKm, name, {})
    all_profiles.append(df); all_summaries.append(sm)

# Grid scan for advanced constraints
for beta in np.linspace(0.90, 1.10, 9):
    for A in [0.35,0.45,0.58,0.70,0.85]:
        for tau in [1.2,1.8,2.5,3.2]:
            for alphaP in alpha_grid:
                params={'beta':float(beta),'A':float(A),'tau':float(tau),'alphaP':float(alphaP)}
                Gm = gamma_p1(levels, beta=beta)
                RKm = response_bounded(levels, A=A, tau=tau, alphaP=alphaP)
                df, sm = metrics_for_sequence(levels, Gm, RKm, 'P1_bounded_grid', params)
                all_profiles.append(df); all_summaries.append(sm)

profiles = pd.concat(all_profiles, ignore_index=True)
summaries = pd.DataFrame(all_summaries)

# Optional external comparison to Test5C-i6 outputs (not used in P1 generation)
test5c_path = Path('/mnt/data/test5c_i6_full/test5C_i6_FULL_IFS_compliance_summary.csv')
comparison = None
if test5c_path.exists():
    t5 = pd.read_csv(test5c_path)
    t5 = t5[['i','R_K_mean','Gamma_mc_mean','K_hat_mean','chi_hat_mean']].rename(columns={
        'R_K_mean':'R_K_Test5C',
        'Gamma_mc_mean':'Gamma_Test5C',
        'K_hat_mean':'K_hat_Test5C',
        'chi_hat_mean':'chi_hat_Test5C'
    })
    base = profiles[profiles['model']=='P1_bounded_baseline'][['i','Gamma','R_K','K_hat','chi_hat']]
    comparison = pd.merge(base,t5,on='i',how='left')
    for col in ['Gamma','chi_hat','K_hat','R_K']:
        tcol = {'Gamma':'Gamma_Test5C','chi_hat':'chi_hat_Test5C','K_hat':'K_hat_Test5C','R_K':'R_K_Test5C'}[col]
        comparison[f'log10_{col}_minus_Test5C'] = np.log10(comparison[col]) - np.log10(comparison[tcol])

# Aggregate verdicts
p1_grid = summaries[summaries['model']=='P1_bounded_grid'].copy()
pass_rate = float(p1_grid['hyper_compliance_flag'].mean()) if len(p1_grid) else None
baseline_summary = summaries[summaries['model']=='P1_bounded_baseline'].iloc[0].to_dict()
control_summary = summaries[summaries['model'].str.startswith('CTRL_')].copy()
control_pass_rate = float(control_summary['hyper_compliance_flag'].mean()) if len(control_summary) else None

# Save outputs
profiles_path = OUT/'test5Cbis_P1_profiles.csv'
summaries_path = OUT/'test5Cbis_P1_model_summary.csv'
profiles.to_csv(profiles_path,index=False)
summaries.to_csv(summaries_path,index=False)
if comparison is not None:
    comparison.to_csv(OUT/'test5Cbis_external_comparison_to_Test5C_i6.csv', index=False)

report = {
    'status':'completed',
    'test_name':'ROSG_Test5Cbis_P1_coupling_compliance_audit',
    'independence_guardrail':'P1 sequences generated without using Test5C/cell-4 graph outputs; Test5C-i6 only used as optional external comparison.',
    'raw_dimensionful_guardrail':'Raw alpha_eff^(i) contains dimension-changing powers and is not directly compared. The audit uses i=1-normalized, dimensionless amplification families.',
    'levels':levels.tolist(),
    'constants':{'hbar':hbar,'G':G,'c':c,'lP_m':lP,'tP_s':tP,'alpha_low_reference':alpha_low},
    'baseline_summary':baseline_summary,
    'p1_grid_n':int(len(p1_grid)),
    'p1_grid_hyper_compliance_pass_rate':pass_rate,
    'control_n':int(len(control_summary)),
    'control_hyper_compliance_pass_rate':control_pass_rate,
    'global_verdict': 'P1_cross_validation_supports_hyper_compliance_trend' if pass_rate and pass_rate>0.95 and baseline_summary['hyper_compliance_flag'] and control_pass_rate < 0.5 else 'conditional_or_inconclusive',
    'reviewer_safe_interpretation': 'This is a cross-validation of the hyper-compliance trend under a P1-normalized iterative amplification model, not a derivation of microscopic gravity or of electromagnetic origin of compliance.',
    'next_recommendation': 'Use Test5C-bis as an independent control before rerunning Test6g with i=6; report trend agreement, not exact numerical equality.'
}
with open(OUT/'test5Cbis_P1_report.json','w') as f:
    json.dump(report,f,indent=2)

# Plots
base = profiles[profiles['model']=='P1_bounded_baseline']
for y, fname, ylabel in [
    ('Gamma','fig_gamma_p1_baseline.png','Gamma_P1'),
    ('R_K','fig_RK_p1_baseline.png','R_K,P1'),
    ('K_hat','fig_Khat_p1_baseline.png','K_hat,P1'),
    ('chi_hat','fig_chihat_p1_baseline.png','chi_hat,P1')
]:
    plt.figure()
    plt.plot(base['i'], base[y], marker='o')
    plt.yscale('log' if y!='R_K' else 'linear')
    plt.xlabel('Iteration i')
    plt.ylabel(ylabel)
    plt.title(f'Test5C-bis P1 baseline: {ylabel}')
    plt.tight_layout()
    plt.savefig(FIG/fname,dpi=160)
    plt.close()

if comparison is not None:
    plt.figure()
    plt.plot(comparison['i'], comparison['chi_hat'], marker='o', label='P1 baseline')
    plt.plot(comparison['i'], comparison['chi_hat_Test5C'], marker='s', label='Test5C-i6')
    plt.yscale('log')
    plt.xlabel('Iteration i')
    plt.ylabel('chi_hat')
    plt.title('External comparison: P1-bis vs Test5C-i6')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG/'fig_external_chi_comparison.png',dpi=160)
    plt.close()

# Manifest
manifest_rows=[]
for path in sorted(OUT.rglob('*')):
    if path.is_file():
        h=hashlib.sha256(path.read_bytes()).hexdigest()
        manifest_rows.append({'path':str(path.relative_to(OUT)),'sha256':h,'bytes':path.stat().st_size})
pd.DataFrame(manifest_rows).to_csv(OUT/'manifest_sha256.csv',index=False)

# Create README
(OUT/'README.md').write_text(textwrap.dedent('''\
# ROSG Test5C-bis — P1 coupling-driven compliance audit

Purpose: provide an independent, P1-normalized cross-check of the hyper-compliance trend observed in Test5C/Test5C-i6.

Key guardrails:
- no cell-4 reconstruction is used to generate P1-bis sequences;
- raw dimensionful powers in alpha_eff^(i) are documented but not compared directly;
- all diagnostics are normalized at i=1;
- response R_K,P1 is constrained to be bounded/saturating;
- null, linear, saturating, and random-order controls are included;
- Test5C-i6 data, if present, is used only as external comparison.

Interpretation: this is a trend-level validation cross-check, not a physical derivation of the coupling or a Mosco/strong-resolvent proof.
'''))

# Zip
zip_path = Path('/mnt/data/ROSG_Test5Cbis_P1_coupling_compliance_audit.zip')
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.rglob('*')):
        if path.is_file():
            z.write(path, arcname=str(Path('ROSG_Test5Cbis_P1_coupling_compliance_audit')/path.relative_to(OUT)))

print(json.dumps(report, indent=2))
print('ZIP', zip_path)


{
  "status": "completed",
  "test_name": "ROSG_Test5Cbis_P1_coupling_compliance_audit",
  "independence_guardrail": "P1 sequences generated without using Test5C/cell-4 graph outputs; Test5C-i6 only used as optional external comparison.",
  "raw_dimensionful_guardrail": "Raw alpha_eff^(i) contains dimension-changing powers and is not directly compared. The audit uses i=1-normalized, dimensionless amplification families.",
  "levels": [
    1,
    2,
    3,
    4,
    5,
    6
  ],
  "constants": {
    "hbar": 1.054571817e-34,
    "G": 6.6743e-11,
    "c": 299792458.0,
    "lP_m": 1.61625502392855e-35,
    "tP_s": 5.391246446661944e-44,
    "alpha_low_reference": 0.0072973525692838015
  },
  "baseline_summary": {
    "model": "P1_bounded_baseline",
    "Gamma_i6_over_i5": 4.000000000000001,
    "R_K_i6_over_i5": 1.0176588541147362,
    "K_hat_i6_over_i5": 0.254414713528684,
    "chi_hat_i6_over_i5": 3.9305902796665686,
    "Gamma_monotone_i3_i6": true,
    "chi_monotone_i3_i6": true,
  

## Interprétation reviewer-safe

Ce notebook ne démontre pas une origine électromagnétique de la compliance et ne constitue pas une preuve Mosco/forte-résolvante. Il teste seulement si un protocole P1 normalisé reproduit qualitativement le régime hyper-compliant : réponse bornée, charge itérative croissante, rigidité décroissante, compliance croissante.